In [1]:
%load_ext autoreload
%autoreload 2

# Load and validate the NF background model

This notebook validates Pascal Janowski's trained normalizing-flow background model against the complete three-month DC4 simulated background reference.

## 1. Setup

In [2]:
from pathlib import Path
import subprocess
import sys

from astropy.io import fits
import h5py
from cosipy.background_estimation.ml import NFBackground
from cosipy.util import fetch_wasabi_file



WARNING FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.



### File paths

Keep the trained checkpoint in `model/` and the complete simulated background in `data/`. These paths are relative to the `notebooks/` directory.

In [3]:
model_path = Path("../model/nfbackground_v2-00.pt")
background_data_path = Path(
    "../data/total_background/Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck.fits.gz"
)

## 2. Validate the trained NF model

Load the version 2.00 checkpoint on CPU. This verifies that `NFBackground` can deserialize the trained model; it does not evaluate or sample the density yet.

In [4]:
nf_background = NFBackground(
    model_path,
    devices=["cpu"],
    density_compile_mode=None,
)

### Checkpoint metadata

`density_input` stores the density-model configuration. The energy and `Phi` cuts define its physical domain. The `bins` entry is the internal rational-quadratic spline resolution, not analysis binning.

In [5]:
density_metadata = nf_background._density_input

print("Available checkpoint keys:")
print(sorted(density_metadata))

Available checkpoint keys:
['bins', 'coefficients', 'context_size', 'hidden_units', 'latent_size', 'menergy_cuts', 'mlp_hidden_layers', 'mlp_hidden_units', 'model_state_dict', 'obs_duration', 'outlocs', 'period', 'phi_cuts', 'residual_blocks', 'slew_duration', 'start_time', 'total_layers', 'total_time']


In [6]:
print("Measured-energy cuts:", density_metadata["menergy_cuts"])
print("Phi cuts:", density_metadata["phi_cuts"])

Measured-energy cuts: [100, 10000]
Phi cuts: [0.0, 3.141592653589793]


## 3. Validate the three-month NF normalization

Run the standalone script in a separate Python process. This avoids the multiprocessing crash caused by calling the NF machinery directly inside the Jupyter kernel. The script integrates the NF rate over the complete DC4 livetime.

In [7]:
subprocess.run(
    [sys.executable, "../src/generate_nf_background.py"],
    check=True,
)


WARNING FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.



NF model loaded successfully
Observation: 1835487300.0 to 1843467255.0
Intervals: 531997



WARNING FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.



Expected NF events in three months: 168,624,174
Poisson-drawn NF events (seed=42): 168,635,152
Time CDF endpoint: 1.000000
Test time range: 1835506429.029 to 1843451466.344
NF test sample shape: (100, 4)
NF test sample minima: [104.0002212524414, 0.10215765237808228, 0.09010939300060272, 0.17371560633182526]
NF test sample maxima: [5836.97998046875, 3.0202105045318604, 6.273352146148682, 2.9571824073791504]
Test events written to: /Users/maura/Documents/COSI/cosi_background/cosipy-background-nf/data/nf_background_test.hdf5
Full NF background already complete: /Users/maura/Documents/COSI/cosi_background/cosipy-background-nf/data/nf_background_3months.hdf5


CompletedProcess(args=['/Users/maura/Documents/COSI/cosi_background/cosipy-background-nf/.venv/bin/python', '../src/generate_nf_background.py'], returncode=0)

### Inspect the generated NF test file

Confirm that the standalone script wrote the five native unbinned COSIpy fields and that each contains the 100 test events.

In [8]:
test_nf_path = Path("../data/NF_background/nf_background_test.hdf5")

with h5py.File(test_nf_path, "r") as file:
    print({name: file[name].shape for name in file})

{'Chi local': (100,), 'Energies': (100,), 'Phi': (100,), 'Psi local': (100,), 'TimeTags': (100,)}


## 4. Inspect the simulated DC4 reference

Download the complete three-month simulated DC4 background only when it is not already present locally.

In [9]:
if background_data_path.exists():
    print(f"Using existing file: {background_data_path}")
else:
    fetch_wasabi_file(
        "COSI-SMEX/DC4/Data/Backgrounds/Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck.fits.gz",
        output=str(background_data_path),
    )

Using existing file: ../data/Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck.fits.gz


In [10]:
size_gb = background_data_path.stat().st_size / 1024**3
print(f"Compressed file size: {size_gb:.2f} GB")

Compressed file size: 14.87 GB


In [11]:
with fits.open(background_data_path, memmap=False, lazy_load_hdus=True) as hdul:
    event_header = hdul[1].header

In [12]:
print(f"Extension: {event_header.get('EXTNAME', 'unnamed')}")
print(f"Events: {event_header['NAXIS2']:,}")
print(f"Columns: {event_header['TFIELDS']}")

Extension: unnamed
Events: 168,648,544
Columns: 11


In [13]:
for index in range(1, event_header["TFIELDS"] + 1):
    print(index, event_header[f"TTYPE{index}"], event_header[f"TFORM{index}"])

1 Energies D
2 TimeTags D
3 Xpointings (glon,glat) 2D
4 Ypointings (glon,glat) 2D
5 Zpointings (glon,glat) 2D
6 Phi D
7 Chi local D
8 Psi local D
9 Distance D
10 Chi galactic D
11 Psi galactic D
